# Clustering validation: complete-linkage vs HDBSCAN

Compares the two same-day clustering candidates from the RAG pipeline redesign (see `docs/rag_redesign/`) against a hand-labeled ground-truth set of article pairs, with a proper train/val/test split so the reported final score isn't contaminated by the same data used to pick parameters.

**Data**: `model_vs_llm_gate_comparison.csv` (title-gate validation set), filtered to articles judged relevant by either gate method (LLM or centroid model), embedded with the same Qwen model used elsewhere in the pipeline.

**Ground truth**: 400 article pairs total, stratified by embedding cosine-similarity band (very high to very low) so the set includes hard cases, not just obvious ones. Each pair was manually read and labeled `1` (same specific story/event) or `0` (different story) by a single labeler (the assistant, working with the user in the design conversation), using a strict rule: **two articles only count as "same" if they report the same specific event/development** — two articles about the same ongoing saga (e.g. the Trump–Canada tariff dispute) but reporting *different* specific developments count as *different*. 83 of 400 pairs are same-story, 317 different-story — split (class-stratified) into:

- `clustering_train.csv` (200 pairs) — used to grid-search each algorithm's parameters
- `clustering_val.csv` (100 pairs) — used to pick the winning algorithm/config, untouched during tuning
- `clustering_test.csv` (100 pairs) — looked at only once, for the final reported score

All three files are human-readable (actual titles, not just indices) so you can audit or correct any label you disagree with.

**Caveats, stated plainly**: this is a single labeler's subjective judgment, not an independently verified or multi-rater-agreed ground truth. It's enough to compare the two algorithms directionally and sanity-check parameter choices — it is not a rigorous, publication-grade benchmark. Treat the final margin as informative, not definitive.

In [7]:
try:
    from google.colab import files
    print("Running in Colab — please upload model_vs_llm_gate_comparison.csv when prompted.")
    print("(Get it fresh from the repo, not a copy from an earlier session, to rule out a stale/corrupted file.)")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"Expected exactly 1 file, got {len(uploaded)}: {list(uploaded.keys())}")
    CSV_PATH = list(uploaded.keys())[0]
    print(f"Uploaded: {CSV_PATH} ({len(uploaded[CSV_PATH])} bytes)")
except ImportError:
    # Not running in Colab — use the file already sitting next to this notebook.
    CSV_PATH = "model_vs_llm_gate_comparison.csv"
    print(f"Not in Colab — using local file: {CSV_PATH}")

Running in Colab — please upload model_vs_llm_gate_comparison.csv when prompted.
(Get it fresh from the repo, not a copy from an earlier session, to rule out a stale/corrupted file.)


Saving model_vs_llm_gate_comparison.csv to model_vs_llm_gate_comparison (1).csv
Uploaded: model_vs_llm_gate_comparison (1).csv (36809733 bytes)


In [8]:
import csv
import json
from collections import defaultdict

import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import HDBSCAN

with open(CSV_PATH, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

relevant = [r for r in rows if r["llm_gate_decision"] == "1" or r["model_decision"] == "1"]
titles = [r["title"] for r in relevant]
langs = [r["language"] for r in relevant]
X = np.array([json.loads(r["embedding"]) for r in relevant], dtype=np.float64)
X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)

print(f"Relevant-by-either articles: {len(relevant)}")
print(f"Embedding matrix shape: {X.shape}")

Relevant-by-either articles: 1511
Embedding matrix shape: (1511, 1024)


In [9]:
def load_pairs(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            out.append((int(r["i"]), int(r["j"]), int(r["label"])))
    return out

train_pairs = load_pairs("clustering_train.csv")
val_pairs = load_pairs("clustering_val.csv")
test_pairs = load_pairs("clustering_test.csv")

for name, pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
    n_pos = sum(1 for *_x, l in pairs if l == 1)
    print(f"{name}: {len(pairs)} pairs ({n_pos} same-story, {len(pairs)-n_pos} different-story)")

train: 200 pairs (42 same-story, 158 different-story)
val: 100 pairs (21 same-story, 79 different-story)
test: 100 pairs (20 same-story, 80 different-story)


In [10]:
def evaluate(labels, pairs):
    """Given a cluster-label array (aligned to `relevant`), score against a list of (i, j, label) pairs.
    Precision/recall/F1 are computed for the 'same cluster' prediction vs the 'same story' ground truth."""
    tp = fp = fn = tn = 0
    for i, j, true_label in pairs:
        pred_same = labels[i] == labels[j]
        if pred_same and true_label == 1:
            tp += 1
        elif pred_same and true_label == 0:
            fp += 1
        elif not pred_same and true_label == 1:
            fn += 1
        else:
            tn += 1
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else float("nan")
    accuracy = (tp + tn) / len(pairs)
    return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}

def singletonize(labels):
    """Turn HDBSCAN's -1 noise label into unique per-point singleton cluster ids,
    matching the 'noise points become singletons' decision made earlier in the design discussion."""
    labels = labels.copy()
    next_id = labels.max() + 1
    for i in range(len(labels)):
        if labels[i] == -1:
            labels[i] = next_id
            next_id += 1
    return labels

## Grid search on TRAIN: complete-linkage agglomerative

Parameter swept: the inconsistency-coefficient cut threshold `t` passed to `scipy.cluster.hierarchy.fcluster(criterion="inconsistent")`. This is the "no fixed distance number" automatic-cut approach decided on earlier — `t` controls how aggressively it cuts, not a raw cosine-distance value. Selection uses `clustering_train.csv` only.

In [11]:
Z = linkage(X_norm, method="complete", metric="cosine")

cl_results = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 2.0]:
    labels = fcluster(Z, t=t, criterion="inconsistent", depth=2)
    metrics = evaluate(labels, train_pairs)
    metrics["t"] = t
    metrics["n_clusters"] = len(set(labels))
    cl_results.append(metrics)

print(f"{'t':>5} {'clusters':>9} {'precision':>10} {'recall':>8} {'f1':>7}")
for r in cl_results:
    print(f"{r['t']:>5} {r['n_clusters']:>9} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1']:>7.3f}")

cl_best = max((r for r in cl_results if r["precision"] == r["precision"]), key=lambda r: r["f1"])
print(f"\nBest complete-linkage on TRAIN: t={cl_best['t']}, F1={cl_best['f1']:.3f}, precision={cl_best['precision']:.3f}, recall={cl_best['recall']:.3f}, {cl_best['n_clusters']} clusters")

    t  clusters  precision   recall      f1
  0.3       975      0.762    0.381   0.508
  0.4       975      0.762    0.381   0.508
  0.5       975      0.762    0.381   0.508
  0.6       975      0.762    0.381   0.508
  0.7       972      0.773    0.405   0.531
  0.8       572      0.676    0.548   0.605
  0.9       525      0.686    0.571   0.623
  1.0       466      0.676    0.595   0.633
  1.1       360      0.643    0.643   0.643
  1.2         1      0.210    1.000   0.347
  1.4         1      0.210    1.000   0.347
  1.6         1      0.210    1.000   0.347
  2.0         1      0.210    1.000   0.347

Best complete-linkage on TRAIN: t=1.1, F1=0.643, precision=0.643, recall=0.643, 360 clusters


## Grid search on TRAIN: HDBSCAN

Parameters swept: `min_cluster_size`, `min_samples`, and `cluster_selection_method` (`eom` vs `leaf` — `leaf` trades persistence for homogeneity, per HDBSCAN's own docs). Noise points are turned into singleton clusters before scoring, matching the earlier design decision. Selection uses `clustering_train.csv` only.

In [12]:
hdb_results = []
for min_cluster_size in [2, 3, 4]:
    for min_samples in [1, 2, 3]:
        for method in ["eom", "leaf"]:
            if min_samples > min_cluster_size:
                continue
            hdb = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples,
                          metric="euclidean", cluster_selection_method=method).fit(X_norm)
            labels = singletonize(hdb.labels_)
            metrics = evaluate(labels, train_pairs)
            metrics.update(min_cluster_size=min_cluster_size, min_samples=min_samples, method=method)
            metrics["n_clusters"] = len(set(labels))
            hdb_results.append(metrics)

print(f"{'mcs':>4} {'ms':>3} {'method':>6} {'clusters':>9} {'precision':>10} {'recall':>8} {'f1':>7}")
for r in sorted(hdb_results, key=lambda r: -r["f1"] if r["f1"] == r["f1"] else -1)[:10]:
    print(f"{r['min_cluster_size']:>4} {r['min_samples']:>3} {r['method']:>6} {r['n_clusters']:>9} {r['precision']:>10.3f} {r['recall']:>8.3f} {r['f1']:>7.3f}")

hdb_best = max((r for r in hdb_results if r["f1"] == r["f1"]), key=lambda r: r["f1"])
print(f"\nBest HDBSCAN on TRAIN: min_cluster_size={hdb_best['min_cluster_size']}, min_samples={hdb_best['min_samples']}, method={hdb_best['method']}, F1={hdb_best['f1']:.3f}, precision={hdb_best['precision']:.3f}, recall={hdb_best['recall']:.3f}, {hdb_best['n_clusters']} clusters")

 mcs  ms method  clusters  precision   recall      f1
   2   1    eom       976      0.744    0.690   0.716
   2   2    eom       976      0.744    0.690   0.716
   3   1    eom       953      0.596    0.810   0.687
   3   2    eom       953      0.596    0.810   0.687
   3   3    eom      1073      0.582    0.762   0.660
   4   1    eom       985      0.507    0.857   0.637
   4   2    eom       985      0.507    0.857   0.637
   4   3    eom      1069      0.515    0.833   0.636
   2   1   leaf      1015      0.719    0.548   0.622
   2   2   leaf      1015      0.719    0.548   0.622

Best HDBSCAN on TRAIN: min_cluster_size=2, min_samples=1, method=eom, F1=0.716, precision=0.744, recall=0.690, 976 clusters


### Select winning algorithm on VAL: complete-linkage vs HDBSCAN

In [15]:
cl_labels_best = fcluster(Z, t=cl_best["t"], criterion="inconsistent", depth=2)
hdb_best_fit = HDBSCAN(min_cluster_size=hdb_best["min_cluster_size"], min_samples=hdb_best["min_samples"],
                        metric="euclidean", cluster_selection_method=hdb_best["method"]).fit(X_norm)
hdb_labels_best = singletonize(hdb_best_fit.labels_)

cl_val = evaluate(cl_labels_best, val_pairs)
hdb_val = evaluate(hdb_labels_best, val_pairs)

print("Validation Set Results:")
print(f"Complete-linkage (t={cl_best['t']}): F1={cl_val['f1']:.3f}, Precision={cl_val['precision']:.3f}, Recall={cl_val['recall']:.3f}")
print(f"HDBSCAN (mcs={hdb_best['min_cluster_size']}, ms={hdb_best['min_samples']}, method={hdb_best['method']}): F1={hdb_val['f1']:.3f}, Precision={hdb_val['precision']:.3f}, Recall={hdb_val['recall']:.3f}")

winning_algo = "complete-linkage" if cl_val['f1'] > hdb_val['f1'] else "hdbscan"
print(f"\nWinning algorithm based on Validation F1: {winning_algo}")

Validation Set Results:
Complete-linkage (t=1.1): F1=0.711, Precision=0.667, Recall=0.762
HDBSCAN (mcs=2, ms=1, method=eom): F1=0.651, Precision=0.636, Recall=0.667

Winning algorithm based on Validation F1: complete-linkage


## Final, only-looked-at-once report on TEST

In [16]:
cl_test = evaluate(cl_labels_best, test_pairs)
hdb_test = evaluate(hdb_labels_best, test_pairs)

print("="*70)
print(f"{'Metric':<15}{'Complete-linkage':<28}{'HDBSCAN':<20}")
print("="*70)
print(f"{'Parameters':<15}{'t='+str(cl_best['t']):<28}{'mcs='+str(hdb_best['min_cluster_size'])+', ms='+str(hdb_best['min_samples'])+', '+hdb_best['method']:<20}")
print(f"{'Precision':<15}{cl_test['precision']:<28.3f}{hdb_test['precision']:<20.3f}")
print(f"{'Recall':<15}{cl_test['recall']:<28.3f}{hdb_test['recall']:<20.3f}")
print(f"{'F1':<15}{cl_test['f1']:<28.3f}{hdb_test['f1']:<20.3f}")
print(f"{'Accuracy':<15}{cl_test['accuracy']:<28.3f}{hdb_test['accuracy']:<20.3f}")
cl_confusion = f"{cl_test['tp']}/{cl_test['fp']}/{cl_test['fn']}/{cl_test['tn']}"
hdb_confusion = f"{hdb_test['tp']}/{hdb_test['fp']}/{hdb_test['fn']}/{hdb_test['tn']}"
print(f"{'TP/FP/FN/TN':<15}{cl_confusion:<28}{hdb_confusion:<20}")

print(f"\nAlgorithm selected on VAL: {winning_algo}")
final = hdb_test if winning_algo == "hdbscan" else cl_test
print(f"Its honest, held-out TEST F1: {final['f1']:.3f} (precision={final['precision']:.3f}, recall={final['recall']:.3f})")

Metric         Complete-linkage            HDBSCAN             
Parameters     t=1.1                       mcs=2, ms=1, eom    
Precision      0.536                       0.652               
Recall         0.750                       0.750               
F1             0.625                       0.698               
Accuracy       0.820                       0.870               
TP/FP/FN/TN    15/13/5/67                  15/8/5/72           

Algorithm selected on VAL: complete-linkage
Its honest, held-out TEST F1: 0.625 (precision=0.536, recall=0.750)


In [20]:
def show_sample(labels, name, n=8, seed=5):
    rng = np.random.default_rng(seed)
    groups = defaultdict(list)
    for idx, l in enumerate(labels):
        groups[l].append(idx)
    multi = {k: v for k, v in groups.items() if len(v) > 1}
    keys = list(multi.keys())
    rng.shuffle(keys)
    print(f"\n=== {name}: {min(n, len(keys))} random multi-member clusters ===")
    for k in keys[:n]:
        idxs = multi[k]
        print(f"-- size {len(idxs)} --")
        for i in idxs[:4]:
            print(f"   [{langs[i]}] {titles[i][:90]}")

show_sample(cl_labels_best, "Complete-linkage (train-selected params)")
show_sample(hdb_labels_best, "HDBSCAN (train-selected params)")


=== Complete-linkage (train-selected params): 8 random multi-member clusters ===
-- size 2 --
   [fr] Hyundai: quand l’IA s’invite dans la grève salariale
   [en] Beyond the AI Layoff Fear: Building Personal Agency in a New Economy
-- size 5 --
   [fr] La Mercedes Classe C aura bientôt une version avec plus de 800 km d'autonomie et un prix 
   [fr] Mercedes Classe C restylée (2026) : à partir de 55 300 €, des prix élevés pour la berline 
   [fr] Prix nouvelle Mercedes Classe C restylée : à partir de 55 300 €, tous les tarifs des versi
   [fr] PHOTOS - Nouvelle Mercedes Classe C restylée : des retouches bien visibles pour la berline
-- size 3 --
   [fr] Norvège : une grève de contrôleurs aériens perturbe le trafic
   [fr] Norvège: grève de contrôleurs aériens, perturbations attendues
   [fr] Une grève des contrôleurs aériens perturbe le trafic en Norvège
-- size 2 --
   [fr] Ford : le Lincoln Nautilus quitte la Chine pour les États-Unis
   [fr] Ford rapatrie le Lincoln Nautilus aux Ét

In [21]:
# show_sample function is already defined in an earlier cell. Calling it with the best labels.
show_sample(cl_labels_best, "Complete-linkage (train-selected params)")
show_sample(hdb_labels_best, "HDBSCAN (train-selected params)")


=== Complete-linkage (train-selected params): 8 random multi-member clusters ===
-- size 2 --
   [fr] Hyundai: quand l’IA s’invite dans la grève salariale
   [en] Beyond the AI Layoff Fear: Building Personal Agency in a New Economy
-- size 5 --
   [fr] La Mercedes Classe C aura bientôt une version avec plus de 800 km d'autonomie et un prix 
   [fr] Mercedes Classe C restylée (2026) : à partir de 55 300 €, des prix élevés pour la berline 
   [fr] Prix nouvelle Mercedes Classe C restylée : à partir de 55 300 €, tous les tarifs des versi
   [fr] PHOTOS - Nouvelle Mercedes Classe C restylée : des retouches bien visibles pour la berline
-- size 3 --
   [fr] Norvège : une grève de contrôleurs aériens perturbe le trafic
   [fr] Norvège: grève de contrôleurs aériens, perturbations attendues
   [fr] Une grève des contrôleurs aériens perturbe le trafic en Norvège
-- size 2 --
   [fr] Ford : le Lincoln Nautilus quitte la Chine pour les États-Unis
   [fr] Ford rapatrie le Lincoln Nautilus aux Ét

## Conclusion

Fill in after running: report the TEST-set winner and its honest F1/precision/recall, note whether it holds up on the manual read (not just the score), and record the chosen production parameters in `docs/rag_redesign/03_clustering.md`.

Remember the ground-truth caveat from the top of this notebook: 400 pairs, one labeler — use this to choose a reasonable default and to compare the two algorithms' *relative* behavior, not as proof of an exact precision/recall number in production.

In [ ]:
cl_labels_best = fcluster(Z, t=cl_best["t"], criterion="inconsistent", depth=2)
hdb_best_fit = HDBSCAN(min_cluster_size=hdb_best["min_cluster_size"], min_samples=hdb_best["min_samples"],
                        metric="euclidean", cluster_selection_method=hdb_best["method"]).fit(X_norm)
hdb_labels_best = singletonize(hdb_best_fit.labels_)

def show_sample(labels, name, n=8, seed=5):
    rng = np.random.default_rng(seed)
    groups = defaultdict(list)
    for idx, l in enumerate(labels):
        groups[l].append(idx)
    multi = {k: v for k, v in groups.items() if len(v) > 1}
    keys = list(multi.keys())
    rng.shuffle(keys)
    print(f"\n=== {name}: {min(n, len(keys))} random multi-member clusters ===")
    for k in keys[:n]:
        idxs = multi[k]
        print(f"-- size {len(idxs)} --")
        for i in idxs[:4]:
            print(f"   [{langs[i]}] {titles[i][:90]}")

show_sample(cl_labels_best, "Complete-linkage (best params)")
show_sample(hdb_labels_best, "HDBSCAN (best params)")

## Conclusion

Fill in after running: report the winning F1/precision/recall, note whether the winner holds up on the manual read (not just the score), and record the chosen production parameters in `docs/rag_redesign/03_clustering.md`.

Remember the ground-truth caveat from the top of this notebook: 120 pairs, one labeler — use this to choose a reasonable default and to compare the two algorithms' *relative* behavior, not as proof of an exact precision/recall number in production.